# Notebook 1 — Dataset Loading, Risk Labelling & Patient-Wise Split

**Project:** ECG RiskNet-XAI — 4-Tier Explainable AI for 12-Lead ECG Risk Prediction  
**Architecture:** InceptionTime + Transformer + SE Attention + Multi-Task Heads  
**Training Dataset:** PTB-XL (12-lead, 500 Hz → resampled 100 Hz, 10 s = 1000 samples)  
**Cross-Dataset Testing:** INCART, Chapman, CPSC, Georgia (all 12-lead)

## Design Decisions
- ✅ **12-lead only** — no 2-lead fallback, no lead reconstruction, no DANN
- ✅ Unified SCP + SNOMED master risk table (no label mismatch)
- ✅ Patient-wise split with ZERO data leakage (verified)
- ✅ Metadata saved alongside signals: age, sex, heart_rate (for Tier 3 fusion)
- ✅ NaN-safe Z-score normalization (std clipped to 1e-8)
- ✅ String-stripped SCP code lookup (no silent .get() failures)
- ✅ Previous data folder cleared before fresh run


## Step 1: Install Required Libraries

In [ ]:
import subprocess, sys

libs = [
    "wfdb", "neurokit2", "scikit-learn", "pandas", "numpy",
    "matplotlib", "seaborn", "scipy", "tqdm", "imbalanced-learn",
    "captum", "shap",
]
for lib in libs:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet"] + lib.split())

print("✅ All libraries installed.")
print("NOTE: Install PyTorch separately with CUDA if needed:")
print("  pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121")


## Step 2: Imports & GPU Verification

In [ ]:
import os, json, warnings, ast, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter
from tqdm import tqdm
import torch

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device      : {device}")
if torch.cuda.is_available():
    print(f"GPU Name    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM        : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU detected — training will be slow.")


## Step 3: Clear Previous Data & Create Save Directory

Deletes old ECG_Project/data so stale .npz / .pt files don't interfere.


In [ ]:
SAVE_DIR = Path("./ECG_Project/data")

if SAVE_DIR.exists():
    shutil.rmtree(SAVE_DIR)
    print("🗑️  Removed previous data folder.")

SAVE_DIR.mkdir(parents=True, exist_ok=True)
print(f"✅ Fresh save directory created: {SAVE_DIR.resolve()}")


## Step 4: Unified Master Disease-to-Risk Table

Single source of truth: SCP codes + SNOMED codes → 4-tier risk.
MIT-BIH mapping is retained only for INCART cross-dataset labelling (same AAMI symbols).


In [ ]:
# Risk Levels: 0=Low  1=Moderate  2=High  3=Critical
RISK_LABELS = {0: "Low", 1: "Moderate", 2: "High", 3: "Critical"}
RISK_ORDER  = ["Low", "Moderate", "High", "Critical"]

SCP_TO_RISK = {
    # NORMAL / LOW
    "NORM": 0, "SR": 0, "SARRH": 0, "EL": 0, "PACE": 0,
    # MILD / MODERATE
    "SBRAD": 1, "STACH": 1, "LAFB": 1, "LPFB": 1, "IRBBB": 1, "ILBBB": 1,
    "LPR": 1, "IVCD": 1, "PAC": 1, "SVPB": 1, "LAE": 1, "RAE": 1,
    "LVS": 1, "ABQRS": 1, "PRC(S)": 1, "NDT": 1, "DIG": 1, "LOWT": 1,
    "PACEAB": 1,
    # HIGH
    "CRBBB": 2, "CLBBB": 2, "RBBB": 2, "LBBB": 2, "PSVT": 2, "WPW": 2,
    "PVC": 2, "BIGU": 2, "TRIGU": 2, "AVB": 2, "BLOCK": 2,
    "ISCAL": 2, "ISCAN": 2, "ISCIN": 2, "ISCLA": 2, "ISC_": 2, "STTC": 2,
    "LVH": 2, "RVH": 2, "LNGQT": 2,
    # CRITICAL
    "AFIB": 3, "AFLT": 3, "VT": 3, "VFIB": 3, "VFLT": 3, "VBRAD": 3,
    "IDIO": 3, "AMI": 3, "IMI": 3, "ALMI": 3, "ASMI": 3, "ILMI": 3,
    "IPMI": 3, "IPLMI": 3, "PMI": 3, "LMI": 3, "ANEUR": 3,
}

SNOMED_TO_RISK = {
    "426783006": 0, "426177001": 1, "427084000": 1, "164889003": 3,
    "164890007": 3, "164909002": 2, "59118001":  2, "270492004": 2,
    "195042002": 2, "27885002":  3, "164947007": 2, "57054005":  3,
    "413444003": 3, "233917008": 2, "251200008": 2, "251173003": 2,
    "428750005": 2, "164873001": 2, "89792004":  2, "11157007":  2,
    "17338001":  2,
}

# AAMI annotation symbol mapping (used for INCART cross-dataset labelling)
MITBIH_TO_RISK = {"N": 0, "S": 1, "V": 2, "F": 2, "Q": 3}

# Lead names in standard clinical order (must match model input channels 0-11)
STANDARD_LEAD_ORDER = ["I", "II", "III", "aVR", "aVL", "aVF",
                        "V1", "V2", "V3", "V4", "V5", "V6"]
NUM_LEADS  = 12
TARGET_LEN = 1000   # 10 s @ 100 Hz
FS_TARGET  = 100

print("✅ Master risk table loaded.")
print(f"   SCP codes    : {len(SCP_TO_RISK)}")
print(f"   SNOMED codes : {len(SNOMED_TO_RISK)}")
print(f"   AAMI codes   : {len(MITBIH_TO_RISK)}")


## Step 5: Configure PTB-XL Path

In [ ]:
# ─── SET THIS TO YOUR LOCAL PTB-XL FOLDER ────────────────────────────────────
# PTB-XL folder must contain ptbxl_database.csv and records500/
PTB_XL_ROOT = r"data/ptb-xl"  # adjust to your system

# Auto-search fallback
ptb_path = Path(PTB_XL_ROOT)
if not ptb_path.exists() or not (ptb_path / "ptbxl_database.csv").exists():
    search_roots = [Path("./"), Path("../"), Path("D:/"), Path("C:/datasets")]
    found = []
    print("Searching for ptbxl_database.csv ...")
    for root in search_roots:
        if not root.exists():
            continue
        try:
            found.extend(root.rglob("ptbxl_database.csv"))
        except PermissionError:
            continue
    if found:
        ptb_path = found[0].parent
        print(f"✅ Found: {ptb_path}")
    else:
        raise FileNotFoundError(
            "ptbxl_database.csv not found. "
            "Download PTB-XL from https://physionet.org/content/ptb-xl/ "
            "and update PTB_XL_ROOT above."
        )
else:
    print(f"✅ PTB_XL_ROOT = '{ptb_path}'")


## Step 6: Load PTB-XL CSV

In [ ]:
import wfdb

meta_csv = ptb_path / "ptbxl_database.csv"
df_meta  = pd.read_csv(meta_csv, index_col="ecg_id")
scp_df   = pd.read_csv(ptb_path / "scp_statements.csv", index_col=0)

print(f"Total records    : {len(df_meta):,}")
print(f"Unique patients  : {df_meta['patient_id'].nunique():,}")
df_meta[["patient_id", "age", "sex", "filename_hr", "scp_codes"]].head()


## Step 7: Assign Risk Labels via Master Table

Uses `code.strip().upper()` to avoid silent lookup failures. Max-risk strategy across all codes with likelihood ≥ 50%.


In [ ]:
def scp_str_to_risk(scp_codes_str):
    try:
        codes = ast.literal_eval(str(scp_codes_str))
    except Exception:
        return None
    max_risk = -1
    for code, likelihood in codes.items():
        if likelihood < 50:
            continue
        key = str(code).strip().upper()
        r = SCP_TO_RISK.get(key, -1)
        if r > max_risk:
            max_risk = r
    return max_risk if max_risk >= 0 else None

df_meta["risk_label"] = df_meta["scp_codes"].apply(scp_str_to_risk)

before = len(df_meta)
df_meta = df_meta.dropna(subset=["risk_label"])
df_meta["risk_label"] = df_meta["risk_label"].astype(int)
after = len(df_meta)

print(f"Records before filtering : {before:,}")
print(f"Records after filtering  : {after:,}  ({before-after} dropped)")
print()
cnt = Counter(df_meta["risk_label"])
for k in range(4):
    n = cnt.get(k, 0)
    print(f"  {RISK_LABELS[k]:10s} ({k}) : {n:6,}  ({100*n/after:.1f}%)")


## Step 8: Extract Metadata Features for Tier 3 Fusion

Age, sex and heart rate are extracted here and saved alongside ECG signals.  
These will be used in the metadata MLP fusion branch of the new architecture.


In [ ]:
# Normalise age: clip to [0,100], fill missing with median
df_meta["age"] = df_meta["age"].clip(0, 100)
age_median = df_meta["age"].median()
df_meta["age"] = df_meta["age"].fillna(age_median)

# sex: 0=Female, 1=Male — fill missing with 0
df_meta["sex"] = df_meta["sex"].fillna(0).astype(int)

# Heart rate: derive from report column if available, else fill with 75 bpm
def extract_hr(row):
    """Try to parse heart rate from scp_codes or default to 75."""
    try:
        codes = ast.literal_eval(str(row["scp_codes"]))
        # PTB-XL doesn't store HR directly; placeholder = 75
    except Exception:
        pass
    return 75.0  # default; replace with actual HR source if available

# Use a fixed default; replace with actual heart rate column if your
# PTB-XL version includes it (some versions have 'heart_rate' column)
if "heart_rate" in df_meta.columns:
    df_meta["hr"] = df_meta["heart_rate"].fillna(75.0).clip(30, 250)
else:
    df_meta["hr"] = 75.0  # placeholder

print("Metadata summary:")
print(df_meta[["age", "sex", "hr"]].describe().round(2))


## Step 9: Patient-Wise Split (Zero Data Leakage)

All ECG records from a single patient are placed in exactly ONE split.  
This is the correct way to split for clinical ECG data.


In [ ]:
from sklearn.model_selection import GroupShuffleSplit

patients = df_meta["patient_id"].values
indices  = np.arange(len(df_meta))

gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
train_idx, test_val_idx = next(gss.split(indices, groups=patients))

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)
rel_val, rel_test = next(gss2.split(
    test_val_idx,
    groups=patients[test_val_idx]
))
val_idx  = test_val_idx[rel_val]
test_idx = test_val_idx[rel_test]

df_train = df_meta.iloc[train_idx]
df_val   = df_meta.iloc[val_idx]
df_test  = df_meta.iloc[test_idx]

# Verify zero patient overlap
assert len(set(df_train["patient_id"]) & set(df_val["patient_id"])) == 0
assert len(set(df_train["patient_id"]) & set(df_test["patient_id"])) == 0
assert len(set(df_val["patient_id"])   & set(df_test["patient_id"])) == 0

print(f"Train : {len(df_train):,} records  |  {df_train['patient_id'].nunique():,} patients")
print(f"Val   : {len(df_val):,} records  |  {df_val['patient_id'].nunique():,} patients")
print(f"Test  : {len(df_test):,} records  |  {df_test['patient_id'].nunique():,} patients")
print("✅ Zero patient overlap confirmed.")


## Step 10: Load & Resample ECG Signals (12-Lead, 1000 Samples)

Loads the 500 Hz high-resolution records, resamples to 100 Hz (1000 samples = 10 s).
Applies per-sample Z-score normalization.


In [ ]:
from scipy.signal import resample as scipy_resample

def load_record(filename, ptb_path, target_len=1000):
    """
    Load a PTB-XL record, resample to target_len samples (12 leads).
    Returns: (1000, 12) float32 array, NaN-safe Z-score normalised.
    """
    rec = wfdb.rdrecord(str(ptb_path / filename))
    sig = rec.p_signal.astype(np.float32)   # (5000, 12) at 500 Hz

    # Resample to target_len
    if sig.shape[0] != target_len:
        sig = scipy_resample(sig, target_len, axis=0)

    # NaN-safe per-sample Z-score normalisation
    mean = sig.mean(axis=0, keepdims=True)
    std  = sig.std(axis=0,  keepdims=True)
    std  = np.where(std < 1e-8, 1e-8, std)
    sig  = (sig - mean) / std
    sig  = np.nan_to_num(sig, nan=0.0, posinf=0.0, neginf=0.0)
    return sig.astype(np.float32)   # (1000, 12)


def build_split(df_split, ptb_path, desc="split"):
    """
    Build X (N,1000,12), y (N,), meta (N,3) arrays from a DataFrame split.
    meta columns: [age, sex, hr]
    """
    signals, labels, metas = [], [], []
    errors = 0
    for _, row in tqdm(df_split.iterrows(), total=len(df_split), desc=desc):
        try:
            sig = load_record(row["filename_hr"], ptb_path)
            signals.append(sig)
            labels.append(int(row["risk_label"]))
            # Metadata: age (normalised /100), sex (0/1), hr (normalised /200)
            metas.append([
                float(row["age"]) / 100.0,
                float(row["sex"]),
                float(row["hr"]) / 200.0,
            ])
        except Exception as e:
            errors += 1
    if errors:
        print(f"  ⚠️  {errors} records failed to load (skipped).")
    return (
        np.stack(signals).astype(np.float32),      # (N, 1000, 12)
        np.array(labels,  dtype=np.int64),
        np.array(metas,   dtype=np.float32),       # (N, 3)
    )


print("Loading training records...")
X_train, y_train, M_train = build_split(df_train, ptb_path, "Train")
print("Loading validation records...")
X_val,   y_val,   M_val   = build_split(df_val,   ptb_path, "Val")
print("Loading test records...")
X_test,  y_test,  M_test  = build_split(df_test,  ptb_path, "Test")

print(f"\nX_train: {X_train.shape}  y_train: {y_train.shape}  M_train: {M_train.shape}")
print(f"X_val  : {X_val.shape}  y_val  : {y_val.shape}")
print(f"X_test : {X_test.shape}  y_test : {y_test.shape}")


## Step 11: Class Distribution Check

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
colors = ["#2ecc71", "#3498db", "#e67e22", "#e74c3c"]

for ax, (name, y) in zip(axes, [("Train", y_train), ("Val", y_val), ("Test", y_test)]):
    cnt = Counter(y)
    bars = ax.bar(
        [RISK_LABELS[k] for k in range(4)],
        [cnt.get(k, 0) for k in range(4)],
        color=colors
    )
    for bar, k in zip(bars, range(4)):
        n = cnt.get(k, 0)
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                f"{100*n/len(y):.1f}%", ha="center", fontsize=9)
    ax.set_title(f"{name} Set  (n={len(y):,})", fontweight="bold")
    ax.set_ylabel("Count")

plt.suptitle("Class Distribution per Split", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(SAVE_DIR / "class_distribution.png", dpi=150)
plt.show()
print("✅ Distribution plot saved.")


## Step 12: Sample ECG Visualisation (One per Risk Level)

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(16, 12))
colors = {0: "#2ecc71", 1: "#3498db", 2: "#e67e22", 3: "#e74c3c"}

for cls, ax in enumerate(axes):
    idxs = np.where(y_train == cls)[0]
    if len(idxs) == 0:
        ax.set_title(f"Risk: {RISK_LABELS[cls]} — No samples found")
        continue
    sample  = X_train[idxs[0]]   # (1000, 12)
    lead_ii = sample[:, 1]        # Lead II (index 1 in standard order)
    t = np.arange(len(lead_ii)) / 100
    ax.plot(t, lead_ii, color=colors[cls], linewidth=0.8)
    ax.set_title(f"Risk: {RISK_LABELS[cls]}  |  Lead II", fontsize=12, fontweight="bold")
    ax.set_xlabel("Time (seconds)")
    ax.set_ylabel("Amplitude (normalised)")
    ax.grid(True, alpha=0.3)

plt.suptitle("Sample ECG Waveforms — Lead II — One per Risk Level", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(SAVE_DIR / "sample_ecgs_by_risk.png", dpi=150)
plt.show()
print("✅ Visualisation saved.")


## Step 13: Save All Splits & Metadata

In [ ]:
# Save splits (X shape: N,1000,12 — transposed to N,12,1000 in NB2 Dataset)
np.savez_compressed(SAVE_DIR / "train_data.npz", X=X_train, y=y_train, meta=M_train)
np.savez_compressed(SAVE_DIR / "val_data.npz",   X=X_val,   y=y_val,   meta=M_val)
np.savez_compressed(SAVE_DIR / "test_data.npz",  X=X_test,  y=y_test,  meta=M_test)

metadata = {
    "risk_labels"    : RISK_LABELS,
    "risk_order"     : RISK_ORDER,
    "scp_to_risk"    : SCP_TO_RISK,
    "snomed_to_risk" : SNOMED_TO_RISK,
    "mitbih_to_risk" : MITBIH_TO_RISK,
    "standard_lead_order": STANDARD_LEAD_ORDER,
    "target_len"     : TARGET_LEN,
    "num_leads"      : NUM_LEADS,
    "fs_target"      : FS_TARGET,
    "meta_features"  : ["age_norm", "sex", "hr_norm"],
    "class_counts"   : {
        "train": {int(k): int(v) for k, v in Counter(y_train).items()},
        "val"  : {int(k): int(v) for k, v in Counter(y_val).items()},
        "test" : {int(k): int(v) for k, v in Counter(y_test).items()},
    }
}

with open(SAVE_DIR / "metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("✅ Metadata saved to metadata.json")
print()
print("NOTEBOOK 1 COMPLETE ✅")
print("=" * 50)
print(f"  {SAVE_DIR}/train_data.npz   (X, y, meta)")
print(f"  {SAVE_DIR}/val_data.npz     (X, y, meta)")
print(f"  {SAVE_DIR}/test_data.npz    (X, y, meta)")
print(f"  {SAVE_DIR}/metadata.json")
print()
print("Next → Notebook 2: Preprocessing, Augmentation & DataLoaders")
